In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import rasterio
import rioxarray
from rasterio import plot
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
import math
from rasterio.mask import mask
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.interpolate import griddata

In [24]:
from utils.site import Site
from utils.tree import Tree
from utils.constants import GHANA_GDF, ETHZ_COCOA_GH_MAP_UINT8_FILEPATH, ETHZ_COCOA_MAP_FILEPATH, SILVER_FOLDERPATH, ETHZ_COCOA_MAP_UINT8_FILEPATH

In [25]:
# Extract Ghana-centered ETHZ predictions
with rasterio.open(ETHZ_COCOA_MAP_UINT8_FILEPATH) as src:
    # 1. Get the window and its specific transform matrix
    minx, miny, maxx, maxy = GHANA_GDF.total_bounds
    window = rasterio.windows.from_bounds(minx, miny, maxx, maxy, src.transform)
    window_transform = rasterio.windows.transform(window, src.transform)

    # 2. Read the data chunk
    data = src.read(1, window=window)

    # 3. Copy the profile and update it completely
    profile = src.profile.copy()

# Update EVERYTHING: size, transform, data type, compression, and nodata
profile.update(
    height=data.shape[0],  # New height from the chunk
    width=data.shape[1],  # New width from the chunk
    transform=window_transform,  # New spatial transform mapping to Ghana
    dtype=data.dtype,
    count=1,
    nodata=255,  # Or your specific data.fill_value
    compress="lzw",
)

print(f"Memory used by Ghana array: {data.nbytes / (1024**3):.2f} GB")

Memory used by Ghana array: 3.36 GB


In [26]:
with rasterio.open(ETHZ_COCOA_GH_MAP_UINT8_FILEPATH, "w", **profile) as dst:
    dst.write(data, 1)

print(f"Dumped: {ETHZ_COCOA_GH_MAP_UINT8_FILEPATH}")

Dumped: /home/aga/Documents/clients/forestero/skymap/skymap/data/silver/cocoa_gh_map_uint8.tif


In [29]:
type(data)

numpy.ndarray